# 🚀 Фабрика нейросетей & Асинхронный Walk-Forward рой PRL/USDT (Google Colab / GPU T4)
Весь тяжёлый код архитектур и обучения вынесен в модульные скрипты  и .
Управляйте параметрами обучения напрямую через аргументы в ячейках ниже.

In [2]:
# 1. Инициализация окружения Colab и принудительное обновление скриптов из репозитория
!nvidia-smi
import os, sys, importlib
if os.path.exists("/content"):
    !rm -rf _repo_tmp && git clone --depth 1 https://github.com/BulletproofBoris/crypto_bot.git _repo_tmp 2>/dev/null || true
    !mkdir -p _tools && cp -rf _repo_tmp/_tools/* _tools/
sys.path.insert(0, os.path.abspath("."))
import _tools.colab_models, _tools.colab_neural_trainer
importlib.reload(_tools.colab_models)
importlib.reload(_tools.colab_neural_trainer)
from _tools.colab_neural_trainer import run_colab_pipeline
print("✅ Свежие модули _tools.colab_models и _tools.colab_neural_trainer загружены!")

/bin/bash: line 1: nvidia-smi: command not found


I0000 00:00:1792502991.593894  172464 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792502991.594676  172464 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792502991.628220  172464 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/restorator/crypto_bot/.venv/lib/python3.14/site-packages/google/api_core/_python_package_support.py:206: FutureWarning: Package google.api_core depends on grpcio, currently installed at version 1.82.1. grpcio < 1.83.0 does not support Post-Quantum Cryptography (

✅ Свежие модули _tools.colab_models и _tools.colab_neural_trainer загружены!


In [3]:
# 2. Конфигурация и запуск Walk-Forward обучения нейросетей (редактируемые параметры)
# Асинхронная сетка окон: H=25 -> 2 окна (375k), H=12 -> 5 окон (210k), H=5 -> 12 окон (75k)
HORIZON = 5                                               # Горизонт H (сек): 5, 12 или 25
WINDOWS = "1,2,3,4,5,6,7,8,9,10,11"                        # Список окон: "all" или "0,1"
ARCHS = "conv1d+gru,causal_cnn,attention,mlp_bottleneck"   # Архитектуры через запятую
RUNS_PER_ARCH = 15                                         # Число независимых ранов на архитектуру
MAX_EPOCHS = 10                                            # Макс. эпох на один ран
BATCH_SIZE = 2048                                          # Размер батча на GPU T4
LR = 1e-2                                                  # Скорость обучения AdamW

df_leaderboard = run_colab_pipeline(
    horizon=HORIZON, windows=WINDOWS, archs=ARCHS,
    runs_per_arch=RUNS_PER_ARCH, max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE, lr=LR
)

I0000 00:00:1792502997.533235  172830 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792502997.570609  172830 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/restorator/crypto_bot/.venv/lib/python3.14/site-packages/google/api_core/_python_package_support.py:206: FutureWarning: Package google.api_core depends on grpcio, currently installed at version 1.82.1. grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In April 2027, Google Cloud Python packages will raise their minimum requirements (including google-api-core, grpcio, and grpcio-status) to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migr

In [ ]:
# 3. Альтернативный запуск через CLI с аргументами командной строки и выгрузка архива
# !python -m _tools.colab_neural_trainer --horizon 12 --windows 0,1,2,3,4 --runs 10 --epochs 10 --batch-size 2048
ZIP_NAME = f"prl_ensemble_pack_h{HORIZON}s.zip"
if os.path.exists(ZIP_NAME):
    !curl -s -F "reqtype=fileupload" -F "time=24h" -F "fileToUpload=@{ZIP_NAME}" https://litterbox.catbox.moe/resources/internals/api.php
    try:
        from google.colab import files
        files.download(ZIP_NAME)
    except Exception:
        pass